# Cheap LLM judge vs. gold-free baselines for NL→FOL faithfulness (Experiment D, analysis stage)

This notebook reproduces the **analysis stage** of *Experiment D*, the baseline/null arm of an evaluation of
**gold-free metrics for natural-language → first-order-logic (NL→FOL) translation**. Each metric gets a sentence and a
candidate FOL formula, with **no gold formula**, and returns a score that should predict whether the formula is
**unfaithful** to the sentence.

**What was scored in the full experiment.** Every item on the shared frozen screen was scored by every baseline:
- **Parse/compile rate**: `parse_fail`.
- **The user's pilot structural metrics**: joint-load conflict with the story (z3), arity/shape clashes, dangling and
  undeclared predicates, and cross-system "rerun" predicate Jaccard.
- **Round-trip (FOL→NL→compare)**: an LLM verbalises the formula, then an NLI model checks entailment in both
  directions against the text (`rt_nli_*`). An mpnet cosine (`rt_embed_cos`) and a re-formalise + z3 equivalence check
  (`rt_reformalise_eq`) are also computed.
- **LLM-as-judge**:
  - `judge_cheap` = gemini-2.5-flash-lite (JSON score 0-100);
  - `judge_cheap2` = gpt-4.1-nano (P(YES) from logprobs);
  - `judge_strong` = gemini-3.1-pro-preview, on a 200+96-item subset only.

  Each judge scores the **original** item and a **nonce-disguised** copy (`_disg`), in which content words are
  replaced by nonce words. The disguise is a contamination control: FOLIO is public, so a judge may have memorised it.

**The labels** come from prover-checked equivalence to corrected gold, or to premises that agreed across systems:
CORRECT / ERROR / UNCERTAIN / UNPARSEABLE.

**What this notebook runs.** Producing the scores needs paid API calls, GPU NLI models and z3 labelling, so those
stages are *not* re-run here. The notebook loads a curated **100-item subset of track L** (Logic-LM gpt-3.5 / gpt-4 /
davinci-003 outputs on FOLIO-dev), with every item's cached oriented metric scores. It then runs the original
`method.py --stage analysis` code (`src/analysis.py` + `src/report.py`) on that subset:
- item-level AUROC / AUPRC, tie rate, threshold metrics and coverage;
- **sentence-clustered bootstrap CIs**;
- paired ΔAUROC vs. the pre-registered bar `judge_cheap_disg`, with a DeLong test;
- the orig-vs-disguised judge delta;
- the **cross-fitted logistic combination** of all cheap baselines (GroupKFold by sentence);
- the T8 sanity checks (shuffled labels, oracle);
- system-level summaries.

Full-run reference numbers (n = 524 track-L items) ship with the data, so the demo estimates can be compared with them.
Every score is **oriented so that higher = more likely unfaithful**.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install (used by the original analysis module for logging)
_pip('loguru==0.7.3')

# numpy, scipy, scikit-learn, pandas, matplotlib — pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# ---- original imports of src/analysis.py / src/report.py / method.py (analysis stage) ----
from __future__ import annotations

import json
import math
import re
import sys
import time
from collections import Counter, defaultdict

import numpy as np
from loguru import logger
from scipy import stats
from sklearn.metrics import average_precision_score, roc_auc_score

# ---- additional imports for the notebook (results table + plots) ----
import pandas as pd
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-eff060-layered-gold-free-checks-for-logic/main/round-1/experiment-4/demo/mini_demo_data.json"
import json
from pathlib import Path

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(data["description"])
print(f"{len(data['examples'])} items;", dict(Counter(e['label'] for e in data['examples'])))
ex = data["examples"][0]
print("\nExample item:\n  text         :", ex["text"], "\n  candidate FOL:", ex["candidate_fol"],
      "\n  label        :", ex["label"], f"({ex['system']})")

## Configuration

These are all the tunable parameters of the analysis. The original values are:
- **all 524 track-L CORRECT/ERROR items** (796 track-L items in total);
- **2,000** cluster-bootstrap resamples with seed 0;
- **5-fold** GroupKFold;
- L2 logistic regression with C = 1.0;
- **20** label shuffles for the T8 sanity check.

The demo data holds 100 items, so `N_ITEMS` can be at most 100. The bootstrap is vectorised, so the original
2,000 resamples fit easily inside the runtime budget.

In [ ]:
N_ITEMS = 40          # items taken from mini_demo_data.json (max 100; the original run used all 796 track-L items)
B_RESAMPLES = 50      # cluster-bootstrap resamples (original: 2000)
SEED = 0              # bootstrap / shuffle seed (original: 0)
N_SPLITS = 5          # GroupKFold folds for the cross-fitted baseline combination (original: 5)
C_LOGREG = 1.0        # L2 logistic-regression strength for the combination (original: 1.0)
N_SHUFFLE = 20        # label permutations in the T8 sanity check (original: 20)

## Metric registry and shared helpers

This cell is copied from `src/analysis.py` (plus `norm` from `src/common.py`). `METRICS` maps each metric name to a
description and a **kind**. The kind decides the pre-registered threshold:

| kind | used for | flag if |
|---|---|---|
| `prob` | judges | 1 − P(faithful) > 0.5 |
| `bin` | binary checks | value = 1 |
| `count` | counts | > 0 |
| `cont` | continuous | above the 90th percentile of track-H CORRECT items |
| `cont_L` | rerun Jaccard | Jaccard < 0.5 |

Two sentinel values mark missing scores:
- `NA`: the metric does not apply to the item by design, so the item is excluded from that metric's evaluation.
- `FAIL`: the metric failed on the item (e.g. judge JSON parse failure, z3 timeout). The item is scored as *flagged*
  (the metric's maximum) and counted against coverage, never silently dropped.

In [ ]:
_QUOTES = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"', "–": "-", "—": "-", "−": "-"})


def norm(s: str) -> str:
    s = (s or "").translate(_QUOTES).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


# name -> (source description, kind) ; kind: prob (judge, thr 0.5) | cont (percentile thr) | bin (>0.5) | count (>0)
METRICS = {
    "parse_fail": ("1 - parses under the shared FOL grammar + z3", "bin"),
    "pilot_joint_conflict": ("story ∪ {cand} UNSAT (z3)", "bin"),
    "pilot_arity_incons": ("#arity clashes with story / within cand", "count"),
    "pilot_shape_incons": ("#arg-kind shape clashes with story", "count"),
    "pilot_dangling": ("fraction of cand symbols absent from story", "cont"),
    "pilot_undeclared": ("fraction of cand predicates not declared (Logic-LM)", "count"),
    "pilot_rerun_jacc": ("1 - cross-system predicate Jaccard (rerun proxy)", "cont_L"),
    "rt_nli_min": ("1 - min(P_entail fwd, bwd)", "cont"),
    "rt_nli_fwd": ("1 - P_entail(text => verbalisation)", "cont"),
    "rt_nli_bwd": ("1 - P_entail(verbalisation => text)", "cont"),
    "rt_nli_contra": ("max P_contradiction", "cont"),
    "rt_nli_min_alt": ("1 - min entail, alt NLI checkpoint", "cont"),
    "rt_embed_cos": ("1 - mpnet cosine(text, verbalisation)", "cont"),
    "rt_reformalise_eq": ("1 - [re-formalised verbalisation ≡ cand modulo vocab]", "bin"),
    "judge_cheap_orig": ("1 - P(faithful), primary cheap judge, original", "prob"),
    "judge_cheap_disg": ("1 - P(faithful), primary cheap judge, DISGUISED (pre-registered bar)", "prob"),
    "judge_cheap2_orig": ("1 - P(YES), secondary judge logprobs, original", "prob"),
    "judge_cheap2_disg": ("1 - P(YES), secondary judge logprobs, disguised", "prob"),
    "judge_strong_orig": ("1 - P(faithful), strong (frontier) judge, original (subset)", "prob"),
    "judge_strong_disg": ("1 - P(faithful), strong (frontier) judge, disguised (subset)", "prob"),
    # secondary: open-weight local judges / local verbaliser (run while the API key was exhausted)
    "judge_local_qwen8b_orig": ("1 - P(faithful), local Qwen3-8B JSON judge, original", "prob"),
    "judge_local_qwen8b_disg": ("1 - P(faithful), local Qwen3-8B JSON judge, disguised", "prob"),
    "judge_local_llama8b_orig": ("1 - P(YES), local Llama-3.1-8B logprob judge, original", "prob"),
    "judge_local_llama8b_disg": ("1 - P(YES), local Llama-3.1-8B logprob judge, disguised", "prob"),
    "judge_local_qwen14b_orig": ("1 - P(faithful), local Qwen3-14B-nf4 judge, original (subset)", "prob"),
    "judge_local_qwen14b_disg": ("1 - P(faithful), local Qwen3-14B-nf4 judge, disguised (subset)", "prob"),
    "rt_nli_min_localverb": ("1 - min NLI entail, local Qwen3-8B verbaliser", "cont"),
    "rt_reformalise_eq_localverb": ("1 - re-formalisation equivalence, local Qwen3-8B verbaliser", "bin"),
}
SUBSET_METRICS = {"judge_strong_orig", "judge_strong_disg", "judge_local_qwen14b_orig", "judge_local_qwen14b_disg"}
NA = "NA"      # not applicable by design (excluded from that metric's evaluation)
FAIL = "FAIL"  # metric failure: scored as the maximum (flagged) and counted against coverage

MAIN_METRICS = [m for m in METRICS if m not in SUBSET_METRICS]


def fmt(a, c=None):
    if a is None:
        return "—"
    return f"{a:.3f}" + (f" [{c[0]:.3f}, {c[1]:.3f}]" if c and c[0] is not None else "")

## Bootstrap and AUROC machinery (verbatim from `src/analysis.py`)

- `fill` maps the sentinels to numbers: `FAIL` becomes the metric's maximum (flagged) and `NA` becomes NaN (excluded).
- `ClusterBoot` draws **sentence clusters** with replacement, where a cluster is all systems' outputs for the same
  normalised sentence. It keeps the resulting multiplicity weights for every resample. All metrics share the *same*
  resamples, which makes paired Δ-AUROC CIs valid.
- `boot_auc` computes a weighted AUROC for all B resamples at once, with ties counted as 0.5.
- `delong` is the paired DeLong test for two correlated AUCs.

In [ ]:
def fill(values: list, maxval: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # -> scores (FAIL -> maxval), applicable mask, fail mask.
    s = np.array([maxval if v == FAIL else (np.nan if v == NA else v) for v in values], dtype=float)
    app = np.array([v != NA for v in values])
    fl = np.array([v == FAIL for v in values])
    return s, app, fl


# ------------------------------------------------------------------ bootstrap machinery
class ClusterBoot:
    # Cluster bootstrap weights: each resample draws clusters with replacement; item weight = cluster multiplicity.

    def __init__(self, clusters: list[str], B: int = B_RESAMPLES, seed: int = SEED):
        self.uc = sorted(set(clusters))
        idx = {c: i for i, c in enumerate(self.uc)}
        self.cid = np.array([idx[c] for c in clusters])
        rng = np.random.default_rng(seed)
        draws = rng.integers(0, len(self.uc), size=(B, len(self.uc)))
        self.W = np.zeros((B, len(self.uc)), dtype=np.int32)
        for b in range(B):
            np.add.at(self.W[b], draws[b], 1)
        self.B = B

    def item_weights(self, b: int) -> np.ndarray:
        return self.W[b][self.cid]


def auc_w(y, s, w=None):
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]
    w = None if w is None else w[ok]
    if w is not None:
        keep = w > 0
        y, s, w = y[keep], s[keep], w[keep]
    if len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, s, sample_weight=w)


def boot_auc(y, s, cb: ClusterBoot):
    # Vectorised weighted AUROC (ties = 0.5) for all B cluster-bootstrap resamples at once.
    ok = ~np.isnan(s)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2:
        return np.full(cb.B, np.nan)
    s_, y_, cid = s[ok], y[ok], cb.cid[ok]
    Wi = cb.W[:, cid].astype(np.float64)                      # B x n item weights
    u, inv = np.unique(s_, return_inverse=True)                # ascending unique scores
    K = len(u)
    onehot = np.zeros((len(s_), K))
    onehot[np.arange(len(s_)), inv] = 1.0
    P = Wi[:, y_ == 1] @ onehot[y_ == 1]                       # B x K positive weight per unique score
    N = Wi[:, y_ == 0] @ onehot[y_ == 0]
    cumN_before = np.cumsum(N, axis=1) - N
    num = (P * (cumN_before + 0.5 * N)).sum(1)
    den = P.sum(1) * N.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(den > 0, num / den, np.nan)
    return out


def ci(a):
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return [None, None]
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]


def tie_rate(y, s):
    ok = ~np.isnan(s)
    pos, neg = s[ok & (y == 1)], s[ok & (y == 0)]
    if len(pos) == 0 or len(neg) == 0:
        return None
    # count tied (pos, neg) pairs via value counts
    cp, cn = Counter(pos.round(9).tolist()), Counter(neg.round(9).tolist())
    ties = sum(cp[v] * cn.get(v, 0) for v in cp)
    return ties / (len(pos) * len(neg))


def delong(y, s1, s2):
    # Paired DeLong test for two correlated AUCs (Sun & Xu 2014 fast version). Returns (auc1, auc2, z, p).
    ok = ~np.isnan(s1) & ~np.isnan(s2)
    y, s1, s2 = y[ok], s1[ok], s2[ok]
    pos, neg = y == 1, y == 0
    m, n = pos.sum(), neg.sum()
    if m < 2 or n < 2:
        return None

    def comps(s):
        x, yv = s[pos], s[neg]
        v10 = np.array([(np.sum(xi > yv) + 0.5 * np.sum(xi == yv)) / n for xi in x])
        v01 = np.array([(np.sum(x > yj) + 0.5 * np.sum(x == yj)) / m for yj in yv])
        return v10.mean(), v10, v01
    a1, v10a, v01a = comps(s1)
    a2, v10b, v01b = comps(s2)
    s10 = np.cov(np.vstack([v10a, v10b]))
    s01 = np.cov(np.vstack([v01a, v01b]))
    S = s10 / m + s01 / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0:
        return float(a1), float(a2), 0.0, 1.0
    z = (a1 - a2) / math.sqrt(var)
    return float(a1), float(a2), float(z), float(2 * (1 - stats.norm.cdf(abs(z))))

## Thresholds and per-set evaluation (verbatim from `src/analysis.py`)

`at_threshold` reports precision, recall, false-alarm rate and balanced accuracy at the pre-registered threshold. It
also re-weights precision to 10% and 25% error prevalence. `evaluate_set` produces the full row for each metric:
- AUROC with its cluster-bootstrap CI;
- AUPRC and prevalence;
- tie rate;
- coverage;
- threshold statistics;
- a `testable` flag, which requires ≥ 50 items per class.

The `thresholds` function is included for completeness. It derives the `cont` thresholds from the 90th percentile of
track-H CORRECT items. Those items are not part of this demo subset, so the notebook uses the **frozen full-run
thresholds** shipped in `mini_demo_data.json`.

In [ ]:
# ------------------------------------------------------------------ thresholds
def thresholds(scores_by_metric: dict, ref_mask_by_metric: dict) -> dict:
    # judges 0.5 on oriented scale; continuous -> 90th pct of oriented score on the reference (track-H CORRECT, dev
    # excluded); binary -> 0.5; counts -> 0 (flag if >0); L-only continuous (rerun jacc) -> 0.5 (Jaccard < 0.5).
    thr = {}
    for m, (_, kind) in METRICS.items():
        if kind == "prob" or kind == "bin":
            thr[m] = 0.5
        elif kind == "count":
            thr[m] = 0.0
        elif kind == "cont_L":
            thr[m] = 0.5
        else:
            s = scores_by_metric[m]
            ref = s[ref_mask_by_metric[m] & ~np.isnan(s)]
            thr[m] = float(np.percentile(ref, 90)) if len(ref) else 0.5
    return thr


def at_threshold(y, s, t):
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]
    f = s > t
    tp, fp = int(np.sum(f & (y == 1))), int(np.sum(f & (y == 0)))
    P, N = int(np.sum(y == 1)), int(np.sum(y == 0))
    rec = tp / P if P else None
    fa = fp / N if N else None
    prec = tp / (tp + fp) if (tp + fp) else None

    def rw(pi):
        if rec is None or fa is None or (rec * pi + fa * (1 - pi)) == 0:
            return None
        return rec * pi / (rec * pi + fa * (1 - pi))
    return {"precision": prec, "recall": rec, "fa": fa, "bal_acc": None if rec is None or fa is None else (rec + 1 - fa) / 2,
            "prec_at_10pct": rw(0.10), "prec_at_25pct": rw(0.25), "n_flagged": int(f.sum())}


def evaluate_set(name, keys, y, table, thr, metrics, cb=None, compute_ci=True):
    # -> ({metric: result}, cb, {metric: boot array}).
    clusters = [table["_cluster"][k] for k in keys]
    if cb is None and compute_ci:
        cb = ClusterBoot(clusters)
    res, boots = {}, {}
    for m in metrics:
        vals = [table[m][k] for k in keys]
        mx = table["_max"][m]
        s, app, fl = fill(vals, mx)
        s_app = np.where(app, s, np.nan)
        n_app = int(app.sum())
        if n_app == 0 or len(np.unique(y[app])) < 2:
            res[m] = {"n_applicable": n_app, "auroc": None, "note": "not applicable / single class"}
            continue
        a = auc_w(y, s_app)
        r = {"n_applicable": n_app, "n_pos": int(np.sum(y[app] == 1)), "n_neg": int(np.sum(y[app] == 0)),
             "prevalence": float(np.mean(y[app])), "auroc": float(a),
             "auprc": float(average_precision_score(y[app], s[app])), "tie_rate": tie_rate(y, s_app),
             "coverage": float(1 - fl[app].mean()), "n_fail": int(fl[app].sum()), "threshold": thr[m],
             **at_threshold(y, s_app, thr[m])}
        r["testable"] = bool(r["n_pos"] >= 50 and r["n_neg"] >= 50)
        if compute_ci:
            ba = boot_auc(y, s_app, cb)
            boots[m] = ba
            r["auroc_ci"] = ci(ba)
        res[m] = r
    return res, cb, boots

## Cross-fitted combination of baselines (verbatim from `src/analysis.py`)

`combo_oof` fits an L2 logistic regression on a feature set of oriented metric scores. Features are standardised on
the training fold, and every feature with missing values gets an extra missing-indicator column. Folds come from
**GroupKFold by sentence**, so all systems' outputs for one sentence land in the same fold. The function returns
**out-of-fold** probabilities, which gives an honest estimate of how far a *combination* of all cheap baselines gets.

In [ ]:
def combo_oof(keys, y, groups, table, feats, n_splits=N_SPLITS, C=C_LOGREG, fold_of=None):
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold
    from sklearn.preprocessing import StandardScaler
    X, cols = [], []
    for f in feats:
        v = [table[f][k] for k in keys]
        s, app, fl = fill(v, table["_max"][f])
        miss = np.isnan(s)
        X.append(np.where(miss, np.nan, s))
        cols.append(f)
        if miss.any():
            X.append(miss.astype(float))
            cols.append(f + "__missing")
    X = np.vstack(X).T
    if fold_of is None:
        gkf = GroupKFold(n_splits=n_splits)
        fold = np.zeros(len(keys), dtype=int)
        for i, (_, te) in enumerate(gkf.split(X, y, groups)):
            fold[te] = i
    else:
        fold = np.array([fold_of[k] for k in keys])
    oof = np.zeros(len(keys))
    for i in range(n_splits):
        tr, te = fold != i, fold == i
        Xtr, Xte = X[tr].copy(), X[te].copy()
        mu = np.nanmean(Xtr, 0)
        mu = np.where(np.isnan(mu), 0, mu)
        sd = np.nanstd(Xtr, 0)
        sd = np.where((sd == 0) | np.isnan(sd), 1, sd)
        Xtr = np.nan_to_num((Xtr - mu) / sd)
        Xte = np.nan_to_num((Xte - mu) / sd)
        clf = LogisticRegression(C=C, max_iter=2000)
        clf.fit(Xtr, y[tr])
        oof[te] = clf.predict_proba(Xte)[:, 1]
    return oof, fold, cols

## Build the score table (adapted from `report.run`)

In the original, `report.run` loads the raw score files (judges, round-trip, pilot) from `results/scores/*.jsonl` and
calls `assemble(...)` to orient them into `{metric: {key: float | NA | FAIL}}`. The demo data already stores the
**assembled, oriented** scores per item, so this cell builds the same `table` directly:
- `_cluster`: the normalised sentence, used as the bootstrap/fold group;
- `_max`: the value a FAIL is mapped to, with the same `max(…, 1.0)` rule as the original.

The thresholds are the frozen full-run values.

In [ ]:
items = data["examples"][:N_ITEMS]
byid = {r["item_id"]: r for r in items}
keys_all = [r["item_id"] for r in items]

# table: metric -> key -> value ; cluster ; max
table = {m: {k: byid[k]["scores"][m] for k in keys_all} for m in METRICS}
table["_cluster"] = {k: norm(byid[k]["text"]) for k in keys_all}
table["_max"] = {}
for m in METRICS:
    vals = [v for v in table[m].values() if v not in (NA, FAIL)]
    table["_max"][m] = max(vals) if vals else 1.0
    if METRICS[m][1] in ("prob", "bin", "cont", "cont_L") and table["_max"][m] < 1.0 and m != "pilot_dangling":
        table["_max"][m] = max(table["_max"][m], 1.0)
# thresholds: frozen full-run values (derived from track-H CORRECT items, dev slice excluded)
thr = {m: data["thresholds_full_run"][m] for m in METRICS}
A = {"thresholds": thr, "metric_definitions": {m: d for m, (d, _) in METRICS.items()}}

print(f"{len(keys_all)} items, {len(set(table['_cluster'].values()))} sentence clusters")
print("labels:", dict(Counter(r['label'] for r in items)), "| systems:", dict(Counter(r['system'] for r in items)))
print("FAIL counts per metric:", {m: sum(v == FAIL for v in table[m].values()) for m in METRICS if any(v == FAIL for v in table[m].values())})

## Evaluation sets and per-metric results (from `report.run`)

These are the track-L views of the original analysis:
- **`L_primary`** (the headline): CORRECT (0) vs ERROR (1).
- **`L_borderline_to_uncertain`**: vocabulary-borderline items dropped.
- **`L_pessimistic_err_or_uncertain`**: UNCERTAIN counted as error.
- **`L_exclude_subst_only`**: "substitution-only" ERRORs dropped. These are likely correct-but-not-equivalent, i.e.
  label noise from a different vocabulary.
- **`L_unparseable_as_error`**: UNPARSEABLE counted as error.

For each set, every metric gets AUROC with its **sentence-cluster bootstrap 95% CI**. All metrics share the same
resamples, so the paired deltas that follow are valid.

In [ ]:
L = [r for r in items]  # all demo items are track L
sets = {
    "L_primary": [(r["item_id"], 1 if r["label"] == "ERROR" else 0) for r in L if r["label"] in ("CORRECT", "ERROR")],
    "L_borderline_to_uncertain": [(r["item_id"], 1 if r["label"] == "ERROR" else 0) for r in L
                                  if r["label"] in ("CORRECT", "ERROR") and not r["vocab_borderline"]],
    "L_pessimistic_err_or_uncertain": [(r["item_id"], 0 if r["label"] == "CORRECT" else 1) for r in L
                                       if r["label"] in ("CORRECT", "ERROR", "UNCERTAIN")],
    "L_exclude_subst_only": [(r["item_id"], 1 if r["label"] == "ERROR" else 0) for r in L
                             if r["label"] in ("CORRECT", "ERROR") and not r["subst_only"]],
    "L_unparseable_as_error": [(r["item_id"], 0 if r["label"] == "CORRECT" else 1) for r in L
                               if r["label"] in ("CORRECT", "ERROR", "UNPARSEABLE")],
}
t0 = time.time()
results, boots, cbs = {}, {}, {}
for sname, ky in sets.items():
    keys = [k for k, _ in ky]
    y = np.array([v for _, v in ky])
    mets = list(METRICS)
    res, cb, bt = evaluate_set(sname, keys, y, table, thr, mets)
    results[sname] = {"n": len(keys), "n_pos": int(y.sum()), "n_neg": int((1 - y).sum()), "metrics": res}
    boots[sname], cbs[sname] = bt, cb
    logger.info(f"set {sname}: n={len(keys)} pos={int(y.sum())}")
A["sets"] = results
logger.info(f"evaluation sets done in {time.time()-t0:.1f}s (B={B_RESAMPLES})")

R = results["L_primary"]
print(f"\nL_primary (n={R['n']}, errors={R['n_pos']}, correct={R['n_neg']})")
print("| metric | AUROC [95% CI] | AUPRC (prev) | tie rate | FA@thr | recall@thr | coverage |")
for m, r in R["metrics"].items():
    if r.get("auroc") is None:
        continue
    print(f"| {m} | {fmt(r['auroc'], r.get('auroc_ci'))} | {r['auprc']:.3f} ({r['prevalence']:.2f}) | "
          f"{(r['tie_rate'] or 0):.2f} | {fmt(r['fa'])} | {fmt(r['recall'])} | {r['coverage']:.3f} |")

## Paired ΔAUROC vs. the pre-registered bar `judge_cheap_disg` (from `report.run`)

The bar is the disguised cheap judge. For each other metric the cell reports ΔAUROC against it, with:
- a paired cluster-bootstrap CI;
- the bootstrap probability that Δ ≤ 0;
- a paired DeLong test on the items where both metrics apply.

In [ ]:
def paired(sname, ref="judge_cheap_disg", others=None):
    ky = sets[sname]
    keys = [k for k, _ in ky]
    y = np.array([v for _, v in ky])
    bt = boots[sname]
    out = {}
    s_ref, a_ref, _ = fill([table[ref][k] for k in keys], table["_max"][ref])
    for m in (others or bt):
        if m == ref or m not in bt or ref not in bt:
            continue
        d = bt[m] - bt[ref]
        s_m, a_m, _ = fill([table[m][k] for k in keys], table["_max"][m])
        both = a_ref & a_m
        dl = delong(y[both], s_m[both], s_ref[both]) if both.sum() > 10 else None
        out[m] = {"delta_auroc": results[sname]["metrics"][m]["auroc"] - results[sname]["metrics"][ref]["auroc"],
                  "ci": ci(d), "p_boot_le0": float(np.mean(d[~np.isnan(d)] <= 0)) if np.any(~np.isnan(d)) else None,
                  "delong": None if dl is None else {"auc_m": dl[0], "auc_ref": dl[1], "z": dl[2], "p": dl[3]},
                  "note": "metric applicable only on a subset" if both.sum() < len(keys) else ""}
    return out
A["paired_vs_judge_cheap_disg"] = {s: paired(s) for s in ("L_primary", "L_exclude_subst_only")}

print("| metric | ΔAUROC | 95% CI | DeLong p |")
for m, v in A["paired_vs_judge_cheap_disg"]["L_primary"].items():
    if v["ci"][0] is None:
        continue
    print(f"| {m} | {v['delta_auroc']:+.3f} | [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}] | "
          f"{(v['delong'] or {}).get('p', float('nan')):.3f} |")

## Contamination: original vs. disguised judge (track-L part of `report.run`)

FOLIO is public, so a judge might recognise memorised gold. The disguise replaces content words with nonce words in
both the sentence and the formula, applied consistently to both. If a judge relies on memorised gold, its AUROC should
**drop** under disguise, and it should drop more on public gold (track H) than on system outputs (track L).

This cell reports the per-set orig → disg AUROC change on track L with a paired bootstrap CI. The full DiD test needs
track H, which is not in the demo subset. In the full run every DiD CI included 0, i.e. no contamination evidence.

In [ ]:
cont = {}
for judge in ("judge_cheap", "judge_cheap2", "judge_local_qwen8b", "judge_local_llama8b"):
    cj = {}
    for sname in ("L_primary",):
        bt = boots[sname]
        o, d = f"{judge}_orig", f"{judge}_disg"
        if o not in bt or d not in bt:
            continue
        delta = bt[d] - bt[o]
        cj[sname] = {"auroc_orig": results[sname]["metrics"][o]["auroc"], "auroc_disg": results[sname]["metrics"][d]["auroc"],
                     "delta_disg_minus_orig": results[sname]["metrics"][d]["auroc"] - results[sname]["metrics"][o]["auroc"],
                     "ci": ci(delta)}
    cont[judge] = cj
A["contamination"] = cont
for j, cj in cont.items():
    for k, v in cj.items():
        print(f"- {j} {k}: AUROC orig {v['auroc_orig']:.3f} → disg {v['auroc_disg']:.3f} "
              f"(Δ {v['delta_disg_minus_orig']:+.3f}, CI [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}])")

## Best baseline combination, cross-fitted (from `report.run`)

The pre-registered feature sets are:
- **S1 structural**: parse + pilot metrics;
- **S2 round-trip**;
- **S3 disguised judges**;
- **S4**: all cheap metrics;
- **S6**: S4 plus the local judges (exploratory).

All feature sets share **one fold assignment**, created by the first fit. The best set among S1–S4 is compared with
`judge_cheap_disg` via paired bootstrap. Taking the max over the four sets makes that number mildly optimistic.

In [ ]:
fs = data["combination_feature_sets"]
S1, S2, S3 = fs["S1_structural"], fs["S2_roundtrip"], fs["S3_judges"]
feature_sets = {"S1_structural": S1, "S2_roundtrip": S2, "S3_judges_disg": S3,
                "S4_all_cheap": S1 + S2 + S3 + ["judge_cheap_orig", "judge_cheap2_orig"],
                "S6_exploratory_S4_plus_local": S1 + S2 + S3 + ["judge_cheap_orig", "judge_cheap2_orig", "judge_local_qwen8b_orig",
                                                                 "judge_local_qwen8b_disg", "judge_local_llama8b_orig",
                                                                 "judge_local_llama8b_disg", "rt_nli_min_localverb"]}
combo = {}
oof_out = {}
folds_out = {}
for tname in ("L_primary",):
    ky = sets[tname]
    keys = [k for k, _ in ky]
    y = np.array([v for _, v in ky])
    groups = [table["_cluster"][k] for k in keys]
    cb = cbs[tname]
    combo[tname] = {}
    fold_ref = None
    for fname, feats in feature_sets.items():
        feats_ok = [f for f in feats if any(table[f][k] != NA for k in keys)]
        oof, fold, cols = combo_oof(keys, y, groups, table, feats_ok, fold_of=fold_ref)
        if fold_ref is None:
            fold_ref = {k: int(f) for k, f in zip(keys, fold)}
        a = auc_w(y, oof)
        ba = boot_auc(y, oof, cb)
        combo[tname][fname] = {"features": feats_ok, "oof_auroc": float(a), "ci": ci(ba), "_boot": ba}
        if tname == "L_primary":
            for k, v in zip(keys, oof):
                oof_out.setdefault(k, {})[f"baseline_combo_{fname}_oof"] = float(v)
    best = max(("S1_structural", "S2_roundtrip", "S3_judges_disg", "S4_all_cheap"), key=lambda f: combo[tname][f]["oof_auroc"])
    combo[tname]["best"] = {"set": best, "oof_auroc": combo[tname][best]["oof_auroc"], "ci": combo[tname][best]["ci"],
                            "note": "max over 4 pre-registered feature sets: mildly optimistic"}
    # best vs judge_cheap_disg paired
    bt = boots[tname]
    d = combo[tname][best]["_boot"] - bt["judge_cheap_disg"]
    combo[tname]["best_minus_judge_cheap_disg"] = {"delta": combo[tname][best]["oof_auroc"] - results[tname]["metrics"]["judge_cheap_disg"]["auroc"],
                                                   "ci": ci(d)}
    if tname == "L_primary":
        for k in keys:
            oof_out[k]["best_baseline_oof"] = oof_out[k][f"baseline_combo_{best}_oof"]
        folds_out = fold_ref
    for f in feature_sets:
        combo[tname][f].pop("_boot", None)
A["baseline_combination"] = combo

cbL = combo["L_primary"]
print("Cross-fitted combinations (track L): " + "; ".join(f"{k}: {fmt(v['oof_auroc'], v['ci'])}" for k, v in cbL.items()
                                                          if isinstance(v, dict) and "oof_auroc" in v and k != "best"))
bm = cbL["best_minus_judge_cheap_disg"]
print(f"best = {cbL['best']['set']}; best − judge_cheap_disg: {bm['delta']:+.3f} [{bm['ci'][0]:.3f}, {bm['ci'][1]:.3f}]")

## T8 sanity checks (from `method.py::_t8_sanity`)

These checks validate the analysis machinery itself:
- Shuffled labels should give AUROC ≈ 0.5.
- An oracle (score = label) should give 1.0.
- Every point estimate should lie inside its bootstrap CI.
- No sentence cluster should be split across folds.

In [ ]:
def _t8_sanity(A_sets_keys, table):
    # T8: shuffled labels ~0.5, oracle = 1.0, CI contains point estimate (checked on L_primary).
    keys, y = A_sets_keys
    rng = np.random.default_rng(0)
    out = {}
    for m in ("judge_cheap_disg", "rt_nli_min", "pilot_joint_conflict"):
        s, app, _ = fill([table[m][k] for k in keys], table["_max"][m])
        s = np.where(app, s, np.nan)
        sh = [auc_w(rng.permutation(y), s) for _ in range(N_SHUFFLE)]
        out[m] = {"shuffled_mean_auroc": float(np.nanmean(sh))}
    out["oracle_auroc"] = float(auc_w(y, y.astype(float)))
    return out


Lp = [(r["item_id"], 1 if r["label"] == "ERROR" else 0) for r in items if r["label"] in ("CORRECT", "ERROR")]
t8 = _t8_sanity(([k for k, _ in Lp], np.array([v for _, v in Lp])), table)
ci_ok = all((r.get("auroc_ci") or [None])[0] is None or (r["auroc_ci"][0] - 1e-9 <= r["auroc"] <= r["auroc_ci"][1] + 1e-9)
            for S in A["sets"].values() for r in S["metrics"].values() if r.get("auroc") is not None)
fold_split = Counter()
for k, f in folds_out.items():
    fold_split[(table["_cluster"][k], f)] += 1
clusters_multi_fold = len({c for c, _ in fold_split}) != len(fold_split)
t8.update({"ci_contains_point": ci_ok, "folds_split_a_sentence": clusters_multi_fold})
A["T8_sanity"] = t8
logger.info(f"T8: {t8}")

## System level (from `report.run`)

The cell compares, per Logic-LM system:
- the true error rate among CORRECT/ERROR items;
- each metric's mean oriented score and flag rate.

It then checks whether ranking the systems by a metric reproduces the true error ranking. There are only three
systems, so this is descriptive.

In [ ]:
sysl = {}
for s in sorted({r["system"] for r in L}):
    its = [r for r in L if r["system"] == s]
    ce = [r for r in its if r["label"] in ("CORRECT", "ERROR")]
    row = {"n": len(its), "label_dist": dict(Counter(r["label"] for r in its)),
           "true_error_rate_CE": sum(r["label"] == "ERROR" for r in ce) / max(1, len(ce)),
           "nonCORRECT_rate_all": sum(r["label"] != "CORRECT" for r in its) / len(its), "metrics": {}}
    for m in MAIN_METRICS:
        s_, app, fl = fill([table[m][r["item_id"]] for r in its], table["_max"][m])
        if app.sum() == 0:
            continue
        row["metrics"][m] = {"mean_score": float(np.nanmean(s_[app])), "flag_rate": float(np.mean(s_[app] > thr[m]))}
    sysl[s] = row
A["system_level"] = {"per_system": sysl, "note": "3 systems only: descriptive, no rank-correlation test"}
order_true = sorted(sysl, key=lambda s: sysl[s]["true_error_rate_CE"])
A["system_level"]["true_error_order"] = order_true
A["system_level"]["metric_order_matches"] = {m: [s for s in sorted(sysl, key=lambda s: sysl[s]["metrics"].get(m, {}).get("mean_score", 0))] == order_true
                                             for m in MAIN_METRICS if all(m in sysl[s]["metrics"] for s in sysl)}

print("| system | true error rate (CORRECT vs ERROR) | judge_cheap_disg flag rate | rt_nli_min mean | parse-fail |")
for sname, v in sysl.items():
    print(f"| {sname} | {v['true_error_rate_CE']:.3f} | {fmt(v['metrics'].get('judge_cheap_disg', {}).get('flag_rate'))} | "
          f"{fmt(v['metrics'].get('rt_nli_min', {}).get('mean_score'))} | {v['label_dist'].get('UNPARSEABLE', 0)}/{v['n']} |")
print("metrics whose system ranking matches the true error ranking:",
      [m for m, ok in A["system_level"]["metric_order_matches"].items() if ok])

## Results: demo subset vs. full run

The table and figure put the demo's `L_primary` AUROCs, from the ≤ 100-item subset, next to the **full-run** values
(n = 524, shipped in the data). Error bars are sentence-cluster bootstrap 95% CIs. The dashed line marks the
pre-registered bar `judge_cheap_disg`.

The subset is small, so the demo CIs are wide. The full-run qualitative picture, which the demo should roughly
reproduce, is:
- The cheap judges (≈ 0.75–0.78) beat round-trip NLI (≈ 0.71).
- The user's pilot structural metrics sit near chance (0.50–0.53), which is a negative result.
- Cross-system rerun Jaccard carries some signal (0.72).
- Re-formalisation equivalence is near chance (0.51).

In [ ]:
ref = data["full_run_reference"]
show = ["parse_fail", "pilot_joint_conflict", "pilot_arity_incons", "pilot_shape_incons", "pilot_dangling", "pilot_undeclared",
        "pilot_rerun_jacc", "rt_nli_min", "rt_nli_min_alt", "rt_embed_cos", "rt_reformalise_eq",
        "judge_cheap_orig", "judge_cheap_disg", "judge_cheap2_orig", "judge_cheap2_disg",
        "judge_local_qwen8b_disg", "judge_local_llama8b_disg", "rt_nli_min_localverb"]
rows = []
for m in show:
    r = results["L_primary"]["metrics"].get(m, {})
    f = ref["L_primary_auroc"].get(m, {})
    rows.append({"metric": m, "demo_auroc": r.get("auroc"), "demo_ci_lo": (r.get("auroc_ci") or [None])[0],
                 "demo_ci_hi": (r.get("auroc_ci") or [None, None])[1], "demo_coverage": r.get("coverage"),
                 "full_auroc": f.get("auroc"), "full_ci_lo": (f.get("ci") or [None])[0], "full_ci_hi": (f.get("ci") or [None, None])[1]})
for k, v in combo["L_primary"].items():
    if isinstance(v, dict) and "oof_auroc" in v and k != "best":
        f = ref["combination_oof_auroc"].get(k, {})
        rows.append({"metric": f"combo:{k}", "demo_auroc": v["oof_auroc"], "demo_ci_lo": v["ci"][0], "demo_ci_hi": v["ci"][1],
                     "demo_coverage": None, "full_auroc": f.get("oof_auroc"), "full_ci_lo": (f.get("ci") or [None])[0],
                     "full_ci_hi": (f.get("ci") or [None, None])[1]})
df = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(f"L_primary: demo n={results['L_primary']['n']} (errors={results['L_primary']['n_pos']}) vs full n={ref['L_primary_n']}")
print(df.round(3).to_string(index=False))

# ---- figure: AUROC with 95% CIs, demo vs full run
d = df.dropna(subset=["demo_auroc"]).reset_index(drop=True)
yy = np.arange(len(d))
fig, ax = plt.subplots(figsize=(9, 0.38 * len(d) + 1.5))
for off, pre, col, lab in ((-0.18, "demo", "#1f77b4", f"demo subset (n={results['L_primary']['n']})"),
                           (0.18, "full", "#ff7f0e", f"full run (n={ref['L_primary_n']})")):
    a = d[f"{pre}_auroc"].astype(float).values
    lo = d[f"{pre}_ci_lo"].astype(float).values
    hi = d[f"{pre}_ci_hi"].astype(float).values
    xerr = np.vstack([np.nan_to_num(a - lo), np.nan_to_num(hi - a)])
    ax.errorbar(a, yy + off, xerr=xerr, fmt="o", color=col, ms=4, capsize=2, label=lab)
ax.axvline(0.5, color="grey", lw=0.8, ls=":")
ax.axvline(results["L_primary"]["metrics"]["judge_cheap_disg"]["auroc"], color="#1f77b4", lw=0.8, ls="--")
ax.axvline(ref["L_primary_auroc"]["judge_cheap_disg"]["auroc"], color="#ff7f0e", lw=0.8, ls="--")
ax.set_yticks(yy)
ax.set_yticklabels(d["metric"])
ax.invert_yaxis()
ax.set_xlabel("AUROC (ERROR vs CORRECT, track L; higher = better detector of unfaithful FOL)")
ax.set_title("Gold-free faithfulness metrics: AUROC with sentence-cluster bootstrap 95% CI")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

# ---- a few items with their judge verdicts
print("\nSample items (judge_cheap_disg = 1 - P(faithful) on the disguised item; flag if > 0.5):")
for r in [x for x in items if x["label"] in ("CORRECT", "ERROR")][:4]:
    print(f"- [{r['label']}] {r['text']}\n    FOL: {r['candidate_fol']}\n    judge_cheap_disg={r['scores']['judge_cheap_disg']} "
          f"rt_nli_min={r['scores']['rt_nli_min']}  judge type={r['judge_cheap_type']}")